In [1]:
# Cell 1: install dependencies (no bitsandbytes -- not supported on Apple Silicon)
!pip install -q -U transformers peft accelerate datasets scikit-learn

In [9]:
import transformers
print(transformers.__version__)

5.7.0


In [2]:
# Cell 2: imports and config
import os
import re
import json
import random
import warnings
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset

warnings.filterwarnings("ignore")

# --- basic config ---
SCORE_COLS  = ("cohesion", "syntax", "vocabulary", "phraseology", "grammar", "conventions")
SCORE_RANGE = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
VAL_SIZE    = 0.1
MAX_LENGTH  = 512
SEED        = 42

# --- LoRA config ---
LORA_R       = 8
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05
LORA_TARGET  = ("q_proj", "k_proj", "v_proj", "o_proj")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# --- training hyperparams ---
EPOCHS     = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4
LR         = 2e-4
OUTPUT_DIR = "./qwen_output"

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# detect device: MPS for Apple Silicon, CUDA for Nvidia, else CPU
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"device: {DEVICE}")

device: mps


In [3]:
# Cell 3: load model
# On Apple Silicon we use float16 -- bitsandbytes 4-bit is not supported on MPS
DTYPE = torch.float16 if DEVICE != "cpu" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map=DEVICE,
    trust_remote_code=True,
)

# enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=list(LORA_TARGET),
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [4]:
# Cell 4: dataset
train_raw = pd.read_csv("train.csv")
test_df   = pd.read_csv("test.csv")

for col in SCORE_COLS:
    train_raw[col] = train_raw[col].astype(float)

train_df, val_df = train_test_split(train_raw, test_size=VAL_SIZE, random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

def get_essay_prompt(essay):
    essay = essay.strip()[:2000]
    prompt_text = (
        "Rate this English essay on 6 dimensions (1.0-5.0 in 0.5 steps).\n"
        'Output ONLY JSON: {"cohesion":x,"syntax":x,"vocabulary":x,'
        '"phraseology":x,"grammar":x,"conventions":x}\n'
        f"Essay:\n{essay}"
    )
    return [{"role": "user", "content": prompt_text}]

class EssayDataset(Dataset):
    def __init__(self, df, tokenizer, is_train=True):
        self.records   = df.to_dict("records")
        self.tokenizer = tokenizer
        self.is_train  = is_train

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row      = self.records[idx]
        messages = get_essay_prompt(row["full_text"])

        input_ids = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=False,
        )

        if self.is_train:
            scores     = {c: round(float(row[c]), 1) for c in SCORE_COLS}
            answer     = json.dumps(scores, separators=(",", ":"))
            answer_ids = self.tokenizer.encode(answer, add_special_tokens=False)

            full_input_ids = input_ids + answer_ids + [self.tokenizer.eos_token_id]
            labels         = [-100] * len(input_ids) + answer_ids + [self.tokenizer.eos_token_id]
        else:
            full_input_ids = input_ids
            labels         = input_ids

        full_input_ids = full_input_ids[:MAX_LENGTH]
        labels         = labels[:MAX_LENGTH]

        return {
            "input_ids":      full_input_ids,
            "labels":         labels,
            "attention_mask": [1] * len(full_input_ids),
        }

train_dataset = EssayDataset(train_df, tokenizer, is_train=True)
val_dataset   = EssayDataset(val_df,   tokenizer, is_train=True)
print(f"train: {len(train_dataset)}, val: {len(val_dataset)}")

train: 3519, val: 392


In [5]:
# Cell 5: training
# MPS does not support bf16; use fp16 instead
use_fp16 = (DEVICE == "mps")
use_bf16 = (DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    fp16=use_fp16,
    bf16=use_bf16,
    gradient_checkpointing=True,
    optim="adamw_torch",  # adamw_bnb_8bit requires bitsandbytes; not available on MPS
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    dataloader_num_workers=0,  # MPS works best with 0 workers
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer, padding="longest", label_pad_token_id=-100
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("starting training...")
trainer.train()

save_path = os.path.join(OUTPUT_DIR, "lora_final")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"saved to {save_path}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


starting training...


Epoch,Training Loss,Validation Loss
1,0.184192,nan
2,0.167593,nan
3,0.165443,nan


saved to ./qwen_output/lora_final


In [8]:
# Cell 6: inference and submission
def clamp_score(v):
    return min(SCORE_RANGE, key=lambda x: abs(x - v))

def parse_json(text):
    try:
        match = re.search(r"\{[^{}]+\}", text, re.DOTALL)
        if match:
            data = json.loads(match.group())
            return {c: clamp_score(float(data.get(c, 3.0))) for c in SCORE_COLS}
    except Exception:
        pass
    return {c: 3.0 for c in SCORE_COLS}

@torch.no_grad()
def generate_score(essay):
    messages = get_essay_prompt(essay)
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt",
        return_dict=False
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=80,
        do_sample=False,
    )
    gen_text = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    return parse_json(gen_text)

print("running inference on test set...")
test_results = []
for i, row in enumerate(test_df.itertuples(), 1):
    res = generate_score(row.full_text)
    test_results.append({"text_id": row.text_id, **res})
    if i % 10 == 0:
        print(f"  {i}/{len(test_df)}")

sub_df = pd.DataFrame(test_results)
sub_df.to_csv("submission.csv", index=False)
print("done. submission.csv saved.")
print(sub_df.head())

running inference on test set...


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


done. submission.csv saved.
        text_id  cohesion  syntax  vocabulary  phraseology  grammar  \
0  0000C359D63E       3.0     2.5         3.0          2.5      2.5   
1  000BAD50D026       2.5     3.0         3.0          2.5      2.5   
2  00367BB2546B       3.5     4.0         3.5          3.5      3.5   

   conventions  
0          2.5  
1          2.5  
2          3.5  
